In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/zeynepdemirta/wustl-ehsm-2020/wustl-ehms-2020_with_attacks_categories.csv


In [2]:
# ===== 1. Find the dataset files =====
from pathlib import Path
import pandas as pd
import numpy as np

candidates = [
    Path("/kaggle/input/datasets/zeynepdemirta/wustl-ehsm-2020"),
    Path("/kaggle/input/wustl-ehsm-2020"),
    Path("/kaggle/input/wustl-ehms-2020"),
    Path("/kaggle/input"),
]

print("=== /kaggle/input tree ===")
root = Path("/kaggle/input")
if root.exists():
    for p in sorted(root.rglob("*")):
        if p.is_file():
            print(f"{p}  ({p.stat().st_size/1024:.1f} KB)")
else:
    print("No /kaggle/input — dataset Add nahi hua")

data_dir = next((c for c in candidates if c.exists()), None)
print("\nUsing:", data_dir)

=== /kaggle/input tree ===
/kaggle/input/datasets/zeynepdemirta/wustl-ehsm-2020/wustl-ehms-2020_with_attacks_categories.csv  (3854.4 KB)

Using: /kaggle/input/datasets/zeynepdemirta/wustl-ehsm-2020


In [3]:
# ===== 2. Load CSV (header/separator auto) =====
csv_files = list(Path("/kaggle/input").rglob("*.csv"))
print("CSV files:", csv_files)
assert csv_files, "CSV nahi mili — Add data check karo"

# WUSTL usually ONE csv; agar multiple hon to sab dikhao
for f in csv_files:
    print(f"\n--- {f.name} ---")
    # try comma, then semicolon
    try:
        df_tmp = pd.read_csv(f, nrows=3)
    except Exception:
        df_tmp = pd.read_csv(f, sep=";", nrows=3)
    print("cols:", list(df_tmp.columns))
    print(df_tmp.head(2))

# main file = largest csv under this dataset
f = max(csv_files, key=lambda x: x.stat().st_size)
print("\n>>> Loading:", f)
df = pd.read_csv(f)
print("shape:", df.shape)
df.head()

CSV files: [PosixPath('/kaggle/input/datasets/zeynepdemirta/wustl-ehsm-2020/wustl-ehms-2020_with_attacks_categories.csv')]

--- wustl-ehms-2020_with_attacks_categories.csv ---
cols: ['Dir', 'Flgs', 'SrcAddr', 'DstAddr', 'Sport', 'Dport', 'SrcBytes', 'DstBytes', 'SrcLoad', 'DstLoad', 'SrcGap', 'DstGap', 'SIntPkt', 'DIntPkt', 'SIntPktAct', 'DIntPktAct', 'SrcJitter', 'DstJitter', 'sMaxPktSz', 'dMaxPktSz', 'sMinPktSz', 'dMinPktSz', 'Dur', 'Trans', 'TotPkts', 'TotBytes', 'Load', 'Loss', 'pLoss', 'pSrcLoss', 'pDstLoss', 'Rate', 'SrcMac', 'DstMac', 'Packet_num', 'Temp', 'SpO2', 'Pulse_Rate', 'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST', 'Attack Category', 'Label']
     Dir        Flgs     SrcAddr     DstAddr  Sport  Dport  SrcBytes  \
0     ->   e          10.0.1.172  10.0.1.150  58059   1111       496   
1     ->   e          10.0.1.172  10.0.1.150  58062   1111       496   

   DstBytes  SrcLoad  DstLoad  ...  Temp  SpO2  Pulse_Rate  SYS  DIA  \
0       186   276914    92305  ...  28.9    

,Dir,Flgs,SrcAddr,DstAddr,Sport,Dport,SrcBytes,DstBytes,SrcLoad,DstLoad,...,Temp,SpO2,Pulse_Rate,SYS,DIA,Heart_rate,Resp_Rate,ST,Attack Category,Label
0,->,e,10.0.1.172,10.0.1.150,58059,1111,496,186,276914.0,92305.0,...,28.9,0,0,0,0,0,0,0.0,normal,0
1,->,e,10.0.1.172,10.0.1.150,58062,1111,496,186,230984.0,76995.0,...,28.9,0,0,0,0,78,17,0.4,normal,0
2,->,e,10.0.1.172,10.0.1.150,58065,1111,496,186,218470.0,72823.0,...,28.9,89,104,0,0,78,17,0.4,normal,0
3,->,e,10.0.1.172,10.0.1.150,58067,1111,496,186,203376.0,67792.0,...,28.9,89,104,0,0,79,17,0.4,normal,0
4,->,e,10.0.1.172,10.0.1.150,58069,1111,496,186,235723.0,78574.0,...,28.9,89,101,0,0,79,17,0.4,normal,0


In [4]:
# ===== 3. Feature report =====
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

print("n rows:", len(df), "| n cols:", df.shape[1])
print("\nCOLUMNS:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:02d}. {c!r}  dtype={df[c].dtype}  nunique={df[c].nunique()}  nan={df[c].isna().sum()}")

n rows: 16318 | n cols: 45

COLUMNS:
01. 'Dir'  dtype=object  nunique=1  nan=0
02. 'Flgs'  dtype=object  nunique=7  nan=0
03. 'SrcAddr'  dtype=object  nunique=1  nan=0
04. 'DstAddr'  dtype=object  nunique=1  nan=0
05. 'Sport'  dtype=object  nunique=16314  nan=0
06. 'Dport'  dtype=int64  nunique=1  nan=0
07. 'SrcBytes'  dtype=int64  nunique=15  nan=0
08. 'DstBytes'  dtype=int64  nunique=14  nan=0
09. 'SrcLoad'  dtype=float64  nunique=8353  nan=0
10. 'DstLoad'  dtype=float64  nunique=8195  nan=0
11. 'SrcGap'  dtype=int64  nunique=1  nan=0
12. 'DstGap'  dtype=int64  nunique=1  nan=0
13. 'SIntPkt'  dtype=float64  nunique=8505  nan=0
14. 'DIntPkt'  dtype=float64  nunique=6094  nan=0
15. 'SIntPktAct'  dtype=float64  nunique=8  nan=0
16. 'DIntPktAct'  dtype=int64  nunique=1  nan=0
17. 'SrcJitter'  dtype=float64  nunique=16212  nan=0
18. 'DstJitter'  dtype=float64  nunique=6046  nan=0
19. 'sMaxPktSz'  dtype=int64  nunique=2  nan=0
20. 'dMaxPktSz'  dtype=int64  nunique=2  nan=0
21. 'sMinPktSz' 

In [5]:
# ===== 4. Guess label + IDs (WUSTL: Label from attacker Src MAC) =====
cols_l = {c.lower(): c for c in df.columns}

label_col = None
for key in ["label", "class", "attack", "target"]:
    if key in cols_l:
        label_col = cols_l[key]
        break

print("LABEL COLUMN:", label_col)
if label_col:
    print(df[label_col].value_counts(dropna=False))
    print("\nnormalized:\n", df[label_col].value_counts(normalize=True).round(4))

# identifiers — IN MODEL MAT DALNA (SrcMAC se label bani hai = leakage)
id_like = []
for c in df.columns:
    cl = c.lower().replace(" ", "").replace("_", "")
    if any(k in cl for k in [
        "srcaddr", "dstaddr", "srcip", "dstip", "srcmac", "dstmac",
        "smac", "dmac", "macaddr", "sport", "dport", "srcport", "dstport",
        "flowno", "seq", "id"
    ]):
        id_like.append(c)

print("\nLIKELY ID / LEAKAGE COLS (drop later):", id_like)

LABEL COLUMN: Label
Label
0    14272
1     2046
Name: count, dtype: int64

normalized:
 Label
0    0.8746
1    0.1254
Name: proportion, dtype: float64

LIKELY ID / LEAKAGE COLS (drop later): ['SrcAddr', 'DstAddr', 'Sport', 'Dport', 'SrcMac', 'DstMac']


In [6]:
# ===== 5. Split: network vs biometric vs categorical =====
BIO_KEYS = [
    "spo2", "pulse", "heart", "resp", "st", "sys", "dia",
    "temp", "ecg", "blood", "hr", "biometric"
]
CAT_KEYS = ["dir", "flgs", "flag", "proto", "state", "srcid", "dstid"]

bio, cat, num, other = [], [], [], []
for c in df.columns:
    if c == label_col:
        continue
    cl = c.lower()
    if any(k in cl for k in CAT_KEYS) or df[c].dtype == "object":
        cat.append(c)
    elif any(k in cl for k in BIO_KEYS):
        bio.append(c)
    elif pd.api.types.is_numeric_dtype(df[c]):
        num.append(c)
    else:
        other.append(c)

print("BIOMETRIC (~8 expected):", bio)
print("CATEGORICAL:", cat)
print("NUMERIC NETWORK:", num)
print("OTHER:", other)
print("\nExpected paper: 35 network + 8 biometric + 1 label = 44 cols")
print("Your total cols:", df.shape[1])

BIOMETRIC (~8 expected): ['DstBytes', 'DstLoad', 'DstGap', 'DstJitter', 'pDstLoss', 'Temp', 'SpO2', 'Pulse_Rate', 'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST']
CATEGORICAL: ['Dir', 'Flgs', 'SrcAddr', 'DstAddr', 'Sport', 'SrcMac', 'DstMac', 'Attack Category']
NUMERIC NETWORK: ['Dport', 'SrcBytes', 'SrcLoad', 'SrcGap', 'SIntPkt', 'DIntPkt', 'SIntPktAct', 'DIntPktAct', 'SrcJitter', 'sMaxPktSz', 'dMaxPktSz', 'sMinPktSz', 'dMinPktSz', 'Dur', 'Trans', 'TotPkts', 'TotBytes', 'Load', 'Loss', 'pLoss', 'pSrcLoss', 'Rate', 'Packet_num']
OTHER: []

Expected paper: 35 network + 8 biometric + 1 label = 44 cols
Your total cols: 45


In [7]:
# ===== 6. Quick stats + class imbalance =====
display(df.describe(include="all").T.head(50))

print("\n=== dtypes ===")
print(df.dtypes.value_counts())

print("\n=== missing % ===")
miss = (df.isna().mean() * 100).sort_values(ascending=False)
print(miss[miss > 0].head(20) if (miss > 0).any() else "no missing")

if label_col:
    print("\n=== numeric means by class ===")
    num_cols = df.select_dtypes(include=np.number).columns.drop(label_col, errors="ignore")
    display(df.groupby(label_col)[num_cols].mean().T.head(25))

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Dir,16318,1,->,16318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Flgs,16318,7,e,15237,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SrcAddr,16318,1,10.0.1.172,16318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DstAddr,16318,1,10.0.1.150,16318,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sport,16318,16314,64273,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dport,16318.0,NaN,NaN,NaN,1111.0,0.0,1111.0,1111.0,1111.0,1111.0,1111.0
SrcBytes,16318.0,NaN,NaN,NaN,496.650264,28.584642,310.0,496.0,496.0,496.0,2298.0
DstBytes,16318.0,NaN,NaN,NaN,187.077706,18.688525,120.0,186.0,186.0,186.0,882.0
SrcLoad,16318.0,NaN,NaN,NaN,211840.633005,79429.880071,0.0,199053.5,236679.0,261557.0,1134000.0
DstLoad,16318.0,NaN,NaN,NaN,71024.35494,45308.106133,507.447,66355.0,78893.0,87193.0,3938000.0



=== dtypes ===
int64      22
float64    15
object      8
Name: count, dtype: int64

=== missing % ===
no missing

=== numeric means by class ===


Label,0,1
Dport,1111.000000,1111.000000
SrcBytes,496.311379,499.014174
DstBytes,186.235426,192.953079
SrcLoad,223851.976078,128054.763827
DstLoad,74617.442340,45960.550755
SrcGap,0.000000,0.000000
DstGap,0.000000,0.000000
SIntPkt,6.853792,39.497467
DIntPkt,3.959571,40.295048
SIntPktAct,1.471408,0.000000


In [8]:
# ===== 7. Ready-to-train drop list (save this) =====
# WUSTL official: Label 0=normal, 1=attack (MITM spoofing + injection)
# Src MAC se label bani — MAC/IP/port drop zaroori

DROP_ALWAYS = [c for c in df.columns if c.lower() in {
    "srcaddr", "dstaddr", "sport", "dport", "srcmac", "dstmac",
    "smacaddr", "dmacaddr", "srcid", "dstid"
} or c in id_like]

X = df.drop(columns=[label_col] + DROP_ALWAYS, errors="ignore")
y = df[label_col].astype(int)

print("Dropped:", DROP_ALWAYS)
print("X:", X.shape, "| y:", y.shape, y.value_counts().to_dict())
print("Remaining features:\n", list(X.columns))

Dropped: ['SrcAddr', 'DstAddr', 'Sport', 'Dport', 'SrcMac', 'DstMac']
X: (16318, 38) | y: (16318,) {0: 14272, 1: 2046}
Remaining features:
 ['Dir', 'Flgs', 'SrcBytes', 'DstBytes', 'SrcLoad', 'DstLoad', 'SrcGap', 'DstGap', 'SIntPkt', 'DIntPkt', 'SIntPktAct', 'DIntPktAct', 'SrcJitter', 'DstJitter', 'sMaxPktSz', 'dMaxPktSz', 'sMinPktSz', 'dMinPktSz', 'Dur', 'Trans', 'TotPkts', 'TotBytes', 'Load', 'Loss', 'pLoss', 'pSrcLoss', 'pDstLoss', 'Rate', 'Packet_num', 'Temp', 'SpO2', 'Pulse_Rate', 'SYS', 'DIA', 'Heart_rate', 'Resp_Rate', 'ST', 'Attack Category']


In [9]:
# ===== 8. Clean Leakage & Zero-Variance Columns =====
# Leakage aur sequence ID drop karein
leakage_extra = ["Attack Category", "Packet_num"]
X = X.drop(columns=leakage_extra, errors="ignore")

# Zero-variance (single unique value) columns remove karein
zero_var_cols = [c for c in X.columns if X[c].nunique() <= 1]
print("Dropping Zero-Variance Columns:", zero_var_cols)
X = X.drop(columns=zero_var_cols)

# Categorical columns check aur One-Hot Encoding (e.g. 'Flgs')
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Encoding Categorical Columns:", cat_cols)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Boolean ko numeric (int) mein convert karein
X = X.astype(float)

print(f"\nFinal Feature Matrix X Shape: {X.shape}")
print(f"Features Count: {X.shape[1]}")

Dropping Zero-Variance Columns: ['Dir', 'SrcGap', 'DstGap', 'DIntPktAct', 'dMinPktSz', 'Trans']
Encoding Categorical Columns: ['Flgs']

Final Feature Matrix X Shape: (16318, 35)
Features Count: 35


In [12]:
# ============================================================
# 1. GENERATE PURE 10-TIMESTEP SEQUENCES ON WUSTL-EHMS-2020
# ============================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEQ_LEN = 10


def generate_pure_sequences(X_scaled, y_array, seq_len=10):
  Xs, ys = [], []
  total_windows = len(X_scaled) - seq_len + 1
  for start in range(total_windows):
    end = start + seq_len
    window_labels = y_array[start:end]
    # Pure-class sequence check (same logic as CICIoMT)
    if np.all(window_labels == window_labels[0]):
      Xs.append(X_scaled[start:end])
      ys.append(window_labels[0])
  return np.asarray(Xs, dtype=np.float32), np.asarray(ys, dtype=np.int32)


# Features aur Targets se sequences generate karein
X_seq_all, y_seq_all = generate_pure_sequences(
    scaler.transform(X), y.values, seq_len=SEQ_LEN
)

# Stratified 80/20 Train-Test Split (Seed 42)
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq_all, y_seq_all, test_size=0.20, random_state=42, stratify=y_seq_all
)

# Further Stratified 80/20 Train-Val Split for training
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_seq,
    y_train_seq,
    test_size=0.20,
    random_state=42,
    stratify=y_train_seq,
)

# Balanced Class Weights compute karein
classes = np.unique(y_tr)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
class_weight_dict = {
    int(c): float(w) for c, w in zip(classes, weights)
}  # Auto balanced

print("=== SEQUENCE DATASET SHAPES ===")
print(f"X_tr : {X_tr.shape} | y_tr : {y_tr.shape}")
print(f"X_val: {X_val.shape} | y_val: {y_val.shape}")
print(f"X_te : {X_test_seq.shape} | y_te : {y_test_seq.shape}")
print("\nClass Weights:", class_weight_dict)

=== SEQUENCE DATASET SHAPES ===
X_tr : (9469, 10, 35) | y_tr : (9469,)
X_val: (2368, 10, 35) | y_val: (2368,)
X_te : (2960, 10, 35) | y_te : (2960,)

Class Weights: {0: 0.5477843341432374, 1: 5.731840193704601}


In [13]:
# ============================================================
# 2. MULTI-SEED (42, 100, 2024) 10-EPOCH LSTM-GRU TRAINING
# ============================================================
import time
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
import tensorflow as tf

SEEDS = [42, 100, 2024]
INPUT_SHAPE = (10, X_tr.shape[2])  # (10, 35)
all_seed_results = []

for s in SEEDS:
  print(f"\n" + "=" * 50)
  print(f"--- Running WUSTL-EHMS-2020 with Seed: {s} ---")
  print("=" * 50)

  tf.keras.utils.set_random_seed(s)

  # Exact Model Architecture from your CICIoMT notebook
  model = tf.keras.Sequential([
      tf.keras.layers.Input(shape=INPUT_SHAPE),
      tf.keras.layers.LSTM(128, return_sequences=True),
      tf.keras.layers.Dropout(0.3),
      tf.keras.layers.GRU(64, return_sequences=False),
      tf.keras.layers.Dropout(0.3),
      tf.keras.layers.Dense(64, activation="relu"),
      tf.keras.layers.Dropout(0.3),
      tf.keras.layers.Dense(
          2, activation="softmax"
      ),  # 2 classes (Normal & Attack)
  ])

  model.compile(
      optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
      loss="sparse_categorical_crossentropy",
      metrics=["accuracy"],
  )

  # Model Training: 10 Epochs
  start_fit = time.time()
  history = model.fit(
      X_tr,
      y_tr,
      validation_data=(X_val, y_val),
      epochs=10,
      batch_size=64,
      class_weight=class_weight_dict,
      verbose=1,
  )

  # Inference & Latency
  start_pred = time.time()
  y_probs = model.predict(X_test_seq, batch_size=64, verbose=0)
  end_pred = time.time()

  y_pred = np.argmax(y_probs, axis=1)
  latency = ((end_pred - start_pred) * 1000) / len(X_test_seq)

  # Metrics calculation
  acc = accuracy_score(y_test_seq, y_pred)
  macro_p = precision_score(y_test_seq, y_pred, average="macro", zero_division=0)
  macro_r = recall_score(y_test_seq, y_pred, average="macro", zero_division=0)
  macro_f1 = f1_score(y_test_seq, y_pred, average="macro", zero_division=0)
  weighted_f1 = f1_score(
      y_test_seq, y_pred, average="weighted", zero_division=0
  )
  auc = roc_auc_score(y_test_seq, y_probs[:, 1])

  # FAR (False Alarm Rate)
  cm = confusion_matrix(y_test_seq, y_pred)
  tn, fp, fn, tp = cm.ravel()
  far = (fp / (fp + tn)) * 100 if (fp + tn) > 0 else 0.0

  all_seed_results.append({
      "Seed": s,
      "Accuracy": acc,
      "Macro_Precision": macro_p,
      "Macro_Recall": macro_r,
      "Macro_F1": macro_f1,
      "Weighted_F1": weighted_f1,
      "ROC_AUC": auc,
      "FAR (%)": far,
      "Latency (ms)": latency,
  })

# ============================================================
# 3. FINAL SUMMARY: MEAN ± STD (Exact Paper Format)
# ============================================================
results_df = pd.DataFrame(all_seed_results)
print("\n" + "=" * 70)
print("INDIVIDUAL SEED RESULTS (WUSTL-EHMS-2020)")
print("=" * 70)
print(results_df.to_string(index=False))

metrics_cols = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Weighted_F1",
    "ROC_AUC",
    "FAR (%)",
    "Latency (ms)",
]

summary_df = pd.DataFrame({
    "Mean": results_df[metrics_cols].mean(),
    "Std": results_df[metrics_cols].std(ddof=1),
})

summary_df["Mean ± Std"] = (
    summary_df["Mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + summary_df["Std"].map(lambda x: f"{x:.4f}")
)

print("\n" + "=" * 70)
print("FINAL 3-SEED RESULTS ON WUSTL-EHMS-2020 (MEAN ± STD)")
print("=" * 70)
print(summary_df[["Mean ± Std"]])


--- Running WUSTL-EHMS-2020 with Seed: 42 ---
Epoch 1/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8839 - loss: 0.4045 - val_accuracy: 0.8678 - val_loss: 0.2976
Epoch 2/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8876 - loss: 0.2842 - val_accuracy: 0.8623 - val_loss: 0.2889
Epoch 3/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8920 - loss: 0.2362 - val_accuracy: 0.8585 - val_loss: 0.2549
Epoch 4/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8944 - loss: 0.2120 - val_accuracy: 0.8704 - val_loss: 0.2441
Epoch 5/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8983 - loss: 0.1963 - val_accuracy: 0.8653 - val_loss: 0.2655
Epoch 6/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9080 - loss: 0.1870 - val_accuracy: 0.8982 - val_loss: 0.1937
Epoch 7/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9195 - loss: 0.1681 - val_accuracy: 0.8974 - val_loss: 0.1921
Epoch 8/10
148/148 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - ac